# GENAI – Modül 1 – Case Study 1



# Görev 1 – LLM’ler Neyi Çözer?

## Senaryo

Bir şirketin elinde:

- 10.000 adet müşteri e-postası
- Günlük destek talepleri
- İç dokümantasyon

bulunmaktadır.

Mevcut sistemde ise:

- Arama işlemleri yavaştır
- E-postaların manuel etiketlenmesi maliyetlidir
- Kural bazlı NLP yaklaşımları yetersiz kalmaktadır


## Soru 1.1 – Klasik NLP ile Çözmeye Çalışırsak Hangi Sınırlara Takılırız?

Klasik NLP burada biraz kelime avcılığı gibi duruyor. Mailde “iade”, “fatura”, “iptal” geçiyorsa o kutuya atıyor. 10 bin mail + her gün yeni talep gelince bu iş tutmaz.

Mesela müşteri “paramı geri istiyorum” ya da “hesabımdan çekilen tutar duruyor” yazıyor. İkisi de iade aslında ama kural listesinde bu cümleler yoksa mail kaçıyor. Aramada da aynı şey: dokümanda “şifre sıfırlama” yazıyor, adam “parolamı unuttum” diye arıyor, sonuç boş. Sistem kelimeye bakıyor, ne demek istediğine bakmıyor.

Etiketleme tarafı da yorucu. Her maili birinin okuması lazım, hem pahalı hem kişiye göre değişiyor. Yarın “kargo gecikmesi” diye yeni bir kutu açsak eski kurallar yetmez, baştan yazıyoruz.

Bir de bağlam yok. “Ödeme” geçen mail şikayet de olabilir teşekkür de. “İptal edin” ile “iptal etmeyin” neredeyse aynı kelimeler, niyet ters. Bir de insanlar kısaltma yazıyor, yazım hatası yapıyor, Türkçe–İngilizce karıştırıyor; kural orada da patlıyor.

Yani arama yavaş kalıyor, etiketleme pahalı kalıyor, kural listesi de durmadan şişiyor.


## Soru 1.2 – LLM Neden Burada Avantajlıdır?

LLM’nin avantajı tam burada: kelimeye takılmıyor, cümlenin ne demek istediğine bakıyor. Aynı modelle maili etiketleyebiliyoruz, talebi özetleyebiliyoruz, iç dokümana bakıp cevap da üretebiliyoruz. Her iş için ayrı kural yazmak zorunda değiliz.

Etiketlemede birkaç örnek göstermek (few-shot) çoğu zaman yetiyor. “Paramı geri istiyorum”un iade talebi olduğunu kurala yazmasak da model bunu anlıyor. 10 bin maili tek tek okumaya gerek kalmıyor.

Aramada da farkı hemen görüyoruz. Biri “parolamı unuttum” diye yazsa bile “şifre sıfırlama” dokümanını bulabiliyor. Destek asistanına da dokümanı verip müşterinin dilinde cevap ürettirebiliyoruz.

Tabii maliyet ve hallüsinasyon ayrı konu, onları da yönetmek lazım. Ama asıl kazanç bağlamı anlaması.


# Görev 2 – Zamir ve Anlam İlişkisi; Transformer Mimarisi

## Görevin Amacı

Bir cümlede geçen belirsiz ifadenin:

- kelime yakınlığına bakılarak mı,
- yoksa cümlenin tamamındaki anlam ilişkilerine bakılarak mı

doğru yorumlanması gerektiğini göstermek.

## Senaryo

> Öğretmen öğrenciyle konuştuktan sonra sınıfa girmedi çünkü o çok sinirliydi.

Bu cümlede geçen **“o”** kelimesi kime karşılık gelmektedir?


## Soru 2.1 – Kelime Bazlı Yorum / Transformer

Sadece kelime yakınlığına bakan bir sistem “o”yu hemen yanındaki isme yapıştırır. Cümlede “o”dan önce gelen isim **sınıf** (`sınıfa girmedi çünkü o`). Biraz daha geniş baksa en yakın kişi **öğrenci** çıkar, çünkü “öğretmen” cümlenin en başında, daha uzak.

Bu yanıltır. Yakın diye doğru olacak diye bir şey yok. Sınıf bir yer, “sinirliydi” bir duygu; sınıfın sinirlenmesi saçma. Öğrenciye bağlamak dil olarak mümkün duruyor ama sistem bunu “çünkü”ye bakarak seçmiyor, sırf mesafe yakın diye seçiyor. Kim sınıfa girmedi, onu da görmüyor.

Yani cümlenin gerisine bakmadan karar veriyor, o yüzden kolay yanılıyor.


## Soru 2.2 – Bağlam / İlişki Bazlı Yorum

Bence “o” burada **öğretmen**. Tek kelimeye bakınca belli olmuyor, cümleyi bütün halinde okumak lazım.

Sınıfa girmeyen kişi öğretmen. Öğrenciyle konuştuktan sonra girmeyen de o. “çünkü” de tam bunu açıklıyor: neden girmedi? Çünkü çok sinirliydi. Sinirli olduğu için bir yere girmemek, eylemi yapan kişiye daha mantıklı geliyor. Sınıfa bağlamak zaten saçma. Öğrenciye de bağlanabilir belki ama “çünkü” asıl öğretmenin neden girmediğini anlatıyor.

Yani ilişki şu tarafta kuruluyor: öğretmen sınıfa girmedi, çünkü sinirliydi.

Transformer da aslında bunu yapıyor. Self-attention sayesinde “o” sadece yanındaki kelimeye bakmıyor; öğretmen, girmedi, çünkü, sinirliydi… hepsine bakabiliyor. Zamiri mesafe ile değil, cümlenin içindeki ilişkiyle çözüyor.


# Görev 3 – Tokenization

## Görevin Amacı

Bir metnin model tarafından nasıl parçalara ayrıldığını (tokenize edildiğini) ve bu parçalanmanın anlam, maliyet ve bağlam uzunluğu üzerindeki etkisini fark etmek.

## Senaryo

Bir şirket, LLM tabanlı bir destek asistanı geliştirmektedir. Sisteme şu müşteri mesajı gider:

> Ödeme tamamlandı fakat hesabım hâlâ ödenmemiş görünüyor.


## Soru 3.1 – Token Kavramı

Bu cümle modele tek parça olarak gitmiyor. Önce küçük parçalara, yani **token**lara bölünüyor. Token bazen kelimenin kendisi, bazen bir ek, bazen birkaç harf, bazen de nokta olabiliyor. Sonra her parça sayıya (vektöre) çevriliyor.

Böyle parçalamak zorundayız çünkü model kelimeyi bizim gibi “okumuyor”. Elinde sabit bir sözlük var. Her Türkçe kelimeyi tek tek tutmaya kalksak sözlük şişer, bir de müşteri “ödenmeiş” diye yanlış yazsa model tamamen kör kalır. “ödenmemiş”i `ö` + `den` + `mem` + `iş` gibi parçalayınca hiç görmediği bir kelimede bile elinde bir şey kalıyor.

Yani modelin gördüğü şey cümle değil, token dizisi. Anlama da, ücret de, context window da hep buradan hesaplanıyor.


## Soru 3.2 – Token Sayısı Üzerine Düşünme

İki cümle de aynı şeyi söylüyor aslında. Bizim için fark yok, model için var çünkü o kelime sayısına değil **token sayısına** bakıyor.

| | İfade | `cl100k_base` |
|---|---|---|
| A | Ödeme tamamlandı fakat hesabım hâlâ ödenmemiş görünüyor. | **26 token** |
| B | Ödeme yapıldı ama sistem hâlâ borç var diyor. | **18 token** |

A, B’den **8 token** daha uzun, yani neredeyse yarı yarıya fazla. A’da kelimeler daha uzun (`tamamlandı`, `ödenmemiş`, `görünüyor`), Türkçe ekler ve `â` de kelimeyi birkaç parçaya bölüyor. Tokenizer bunları tek parça tutmuyor.

Bu neden önemli? API token başına para alıyor. 10 bin müşteri mesajında A gibi yazılmış mailler B’ye göre daha pahalıya geliyor. Bir de context window daha çabuk doluyor, yani aynı pencereye daha az şey sığdırıyoruz.

Anlam aynı, model daha uzun bir girdi görüyor. Aşağıda tokenizer’la parçalarına ayırdım.


In [1]:
# Aynı anlam, farklı token sayısı
# yoksa: python3 -m pip install tiktoken

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-3.5 / GPT-4 tokenizer

ornek_a = "Ödeme tamamlandı fakat hesabım hâlâ ödenmemiş görünüyor."
ornek_b = "Ödeme yapıldı ama sistem hâlâ borç var diyor."


def token_ozeti(metin, etiket):
    token_ids = enc.encode(metin)
    parcalar = [enc.decode([t]) for t in token_ids]
    print(f"{etiket}")
    print(f"  Metin        : {metin}")
    print(f"  Kelime sayısı: {len(metin.split())}")
    print(f"  Karakter     : {len(metin)}")
    print(f"  Token sayısı : {len(token_ids)}")
    print(f"  Token parçaları: {parcalar}")
    print()
    return len(token_ids)


n_a = token_ozeti(ornek_a, "A")
n_b = token_ozeti(ornek_b, "B")
print(f"Fark: A, B'den {n_a - n_b} token daha fazla ({(n_a / n_b - 1) * 100:.0f}% daha uzun).")

A
  Metin        : Ödeme tamamlandı fakat hesabım hâlâ ödenmemiş görünüyor.
  Kelime sayısı: 7
  Karakter     : 56
  Token sayısı : 26
  Token parçaları: ['Ö', 'd', 'eme', ' tam', 'am', 'land', 'ı', ' f', 'ak', 'at', ' hes', 'ab', 'ım', ' h', 'â', 'l', 'â', ' ö', 'den', 'mem', 'iş', ' gör', 'ün', 'ü', 'yor', '.']

B
  Metin        : Ödeme yapıldı ama sistem hâlâ borç var diyor.
  Kelime sayısı: 8
  Karakter     : 45
  Token sayısı : 18
  Token parçaları: ['Ö', 'd', 'eme', ' yapı', 'ld', 'ı', ' ama', ' sistem', ' h', 'â', 'l', 'â', ' bor', 'ç', ' var', ' di', 'yor', '.']

Fark: A, B'den 8 token daha fazla (44% daha uzun).


## Soru 3.3 – Prompt Tasarımı ile Token İlişkisi

Prompt yazarken token sayısını boşuna düşünmüyoruz, çünkü fatura buradan çıkıyor. Hem verdiğimiz yazı hem modelin cevabı token olarak sayılıyor. “Lütfen çok detaylı bir şekilde açıklar mısınız…” gibi cümleler kibar duruyor ama anlama pek bir şey katmıyor, sadece ücreti şişiriyor. Destek asistanı günde binlerce kez çalışıyorsa bu fark ciddiye biner.

Bir de context window var. Model aynı anda her şeyi göremez. Prompt’a gereksiz talimat, eski sohbet, bütün dokümanı tıkıştırınca asıl müşteri mesajı pencereden düşebiliyor. Kısa tutunca gerçekten gereken şeye yer kalıyor.

Fazla kelime talimatı da bozuyor aslında. Model ne istediğimizi kaybediyor. İyi prompt kime yazıyoruz, ne istiyoruz, çıktı nasıl olsun, bunları uzatmadan söylüyor.


# Görev 4 – Prompt Engineering

## Görevin Amacı

Aynı verinin, farklı şekilde yazılmış prompt’larla nasıl farklı çıktılar ürettiğini gözlemlemek.

## Senaryo

Şirket içi dokümantasyonda şu metin vardır:

> Yeni sürüm ile birlikte kullanıcı şifreleri artık minimum 10 karakter olmalıdır.
> Ayrıca en az bir büyük harf ve bir rakam içermesi zorunludur.
> Bu kurallara uymayan şifreler sistem tarafından kabul edilmeyecektir.

Bu metnin farklı ekipler için farklı amaçlarla özetlenmesi istenmektedir.


## Soru 4.1 – Genel ve Belirsiz Prompt

**Prompt:**

```text
Bu metni özetle.
```

**Tipik çıktı:**

> Yeni sürümde şifreler en az 10 karakter olmalı. Bir büyük harf ve bir rakam da şart. Uymayanlar sistemde kabul edilmiyor.

Çıktı yanlış değil, kısaltmış. Ama kime yazıldığı belli değil. Kullanıcı mı okuyacak, yazılımcı mı, destek ekibi mi? Ona göre dil de değişir, detay da.

Bu haliyle biraz boş kalıyor. “Özetle” deyince model kendi bildiği genel özeti basıyor. Madde madde mi olsun, sade mi olsun, validasyon kuralı mı çıksın, hiçbiri yok. Aynı prompt’u her ekibe versen hepsi birbirine benzeyen bir metin alır. İşe yarar bir şey istiyorsak prompt’ta kime yazdığımızı ve ne için istediğimizi söylemek lazım.


## Soru 4.2 – Hedef Odaklı Prompt Yazımı

### a) Teknik olmayan son kullanıcılar için

```text
Bunu normal kullanıcının anlayacağı şekilde, sade Türkçe ile özetle.
Şifre belirlerken nelere dikkat etmesi gerektiğini söyle.
Teknik terim kullanma, 3-4 cümleyi geçme.

Metin:
Yeni sürüm ile birlikte kullanıcı şifreleri artık minimum 10 karakter olmalıdır.
Ayrıca en az bir büyük harf ve bir rakam içermesi zorunludur.
Bu kurallara uymayan şifreler sistem tarafından kabul edilmeyecektir.
```

**Örnek çıktı:**

> Artık şifre belirlerken en az 10 karakter kullanmanız gerekiyor. İçinde bir tane büyük harf ve bir tane de rakam olsun. Bunlara uymayan şifreyi sistem kabul etmiyor.

### b) Yazılım geliştirme ekibi için

```text
Bunu yazılım ekibi için madde madde özetle.
Minimum uzunluk, zorunlu karakterler ve uymayan şifrenin reddedileceğini yaz.
Kullanıcıya konuşur gibi yazma, kodlarken işimize yarasın.

Metin:
Yeni sürüm ile birlikte kullanıcı şifreleri artık minimum 10 karakter olmalıdır.
Ayrıca en az bir büyük harf ve bir rakam içermesi zorunludur.
Bu kurallara uymayan şifreler sistem tarafından kabul edilmeyecektir.
```

**Örnek çıktı:**

> - min 10 karakter
> - en az 1 büyük harf + 1 rakam
> - uymayan şifre reddedilecek
> - kontrolü backend’de tutmak lazım
> - test: 9 karakterlik, büyük harfsiz ve rakamsız örnekleri de dene

İkisi de aynı metinden çıktı. Biri kullanıcıya “ne yapayım” diyor, öteki geliştiriciye “neyi kodlarım” diyor. Dil, detay ve amaç değişince çıktı da değişiyor.


## Soru 4.3 – Karşılaştırmalı Değerlendirme

Kaynak metin hiç değişmedi, değişen tek şey prompt. Model de ona göre yazıyor zaten. “Özetle” deyince eline genel bir özet veriyor. “Kullanıcıya sade anlat, 3-4 cümle” deyince dil yumuşuyor, kısa kalıyor. “Geliştiriciye madde madde yaz” deyince aynı bilgi kural listesine dönüyor.

Yani prompt aslında verinin kendisi değil, modele “bunu kime, nasıl anlat” dediğimiz yer. Ne kadar açık söylersek o kadar az saçmalıyor. 4.1’de herkese aynı metin çıkıyor, 4.2’de iki ayrı iş çıktısı var. Modeli baştan eğitmeden, sadece cümleyi değiştirerek bunu yapabiliyoruz.
